In [3]:
print("hello")

hello


In [4]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

def loadPdfFilles(path):
    # allow either relative (to workspace root) or absolute paths
    path_obj = Path(path)
    if not path_obj.is_absolute():
        # assume notebook is running from the research folder
        base = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
        path_obj = base / path_obj

    loader=DirectoryLoader(
        str(path_obj),
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents=loader.load()
    return documents

c:\Users\vivobook\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# loadPdfFilles with relative path; function will resolve to project root
extractedData = loadPdfFilles("data")

Multiple definitions in dictionary at byte 0x5f1b5 for key /Info
Multiple definitions in dictionary at byte 0x5f1c1 for key /Info
Multiple definitions in dictionary at byte 0x5f1cd for key /Info
Multiple definitions in dictionary at byte 0x4560c for key /Info
Multiple definitions in dictionary at byte 0x45618 for key /Info
Multiple definitions in dictionary at byte 0x45624 for key /Info


In [6]:
extractedData # list of pages 

[Document(metadata={'producer': 'macOS Version 11.3.1 (assemblage 20E241) Quartz PDFContext', 'creator': 'HP Scan', 'creationdate': '2025-02-03T13:44:59+00:00', 'moddate': '2026-02-07T21:25:46+01:00', 'source': 'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject2\\chatBotJuridiquesWithRAG-\\data\\AMTJ-parents.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'macOS Version 11.3.1 (assemblage 20E241) Quartz PDFContext', 'creator': 'HP Scan', 'creationdate': '2025-02-03T13:44:59+00:00', 'moddate': '2026-02-07T21:25:46+01:00', 'source': 'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject2\\chatBotJuridiquesWithRAG-\\data\\AMTJ-parents.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content=''),
 Document(metadata={'producer': 'macOS Version 11.3.1 (assemblage 20E241) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260123110706Z00'00'", 'moddate': "D:20260123110706Z00'00'",

In [7]:
len(extractedData) #nbr of pages

24

turning pdfs into images and extract the text and turn it to json

In [8]:
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import json
import os
import re  # <--- زدنا هادي باش نقيو النص

# ==========================================
# 1. BULLETPROOF PATHING
# ==========================================
current_path = Path.cwd()

# If running from inside the 'research' folder, go up one level
if current_path.name == "research":
    BASE_DIR = current_path.parent 
else:
    # Failsafe: Hardcoded absolute path
    BASE_DIR = Path(r"C:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject2\chatBotJuridiquesWithRAG-")

PDF_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "data" / "pages"
OCR_OUT_DIR = BASE_DIR / "artifacts" / "ocr"

# ==========================================
# 2. CONFIGURATION
# ==========================================
DPI = 300
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SLICE_HEIGHT = 240 

# sanity checks for external dependencies
if not Path(POPPLER_PATH).exists():
    print(f"⚠️  Poppler path {POPPLER_PATH} does not exist. PDF conversion may fail.")

if not Path(pytesseract.pytesseract.tesseract_cmd).exists():
    print(f"⚠️  Tesseract binary {pytesseract.pytesseract.tesseract_cmd} not found. OCR may fail.")

# ==========================================
# 3. TEXT CLEANING FUNCTION (هنا فين كاين السر)
# ==========================================
def clean_text(text):
    if not text:
        return ""
    # 1. تبديل الرجوع للسطر و Tab بفراغ عادي
    text = re.sub(r'[\n\t\r]+', ' ', text)
    # 2. مسح الرموز والإيموجيز (نخليو غير الحروف، الأرقام، والترقيم الأساسي)
    text = re.sub(r'[^\w\s.,;:!؟،؛"\'\(\)\-/]', '', text)
    # 3. مسح الفراغات الزايدة (باش مايبقاش ليسباس كبير بين الكلمات)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_patches(img, slice_height=SLICE_HEIGHT):
    w, h = img.size
    patches = []
    coords = []
    for top in range(0, h, slice_height):
        box = (0, top, w, min(top + slice_height, h))
        patch = img.crop(box)
        patches.append(patch)
        coords.append(box)
    return patches, coords

def ocr_patches(patches):
    texts = []
    total = len(patches)
    for i, patch in enumerate(patches):
        # Progress indicator so you know it's not frozen
        print(f"      -> Extracting text from patch {i+1}/{total}...", end="\r")
        text = pytesseract.image_to_string(patch, lang='ara', config='--psm 6')
        
        # نطبقو التنظيف على النص المخرج من الصورة
        cleaned = clean_text(text)
        texts.append(cleaned)
        
    print() # Move to the next line after finishing the patches
    return texts

def process_all_pdfs():
    print(f"--- Working Directory: {BASE_DIR} ---")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OCR_OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    pdfs = list(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Error: No PDFs found in {PDF_DIR}")
        return

    print(f"\n--- STEP 1: CONVERTING {len(pdfs)} PDFs TO IMAGES ---")
    for pdf_path in pdfs:
        doc_id = pdf_path.stem
        doc_out = OUT_DIR / doc_id
        doc_out.mkdir(parents=True, exist_ok=True)
        try:
            print(f"⏳ Converting {pdf_path.name}...")
            # باش نربحو الوقت: مابغيتيش يعاود يحول الصور يلا كانو ديجا كاينين؟
            if not list(doc_out.glob("*.png")):
                pages = convert_from_path(pdf_path, dpi=DPI, fmt="png", poppler_path=POPPLER_PATH)
                for i, page in enumerate(pages, start=1):
                    page_path = doc_out / f"page_{i:03d}.png"
                    if not page_path.exists():
                        page.save(page_path, "PNG")
                print(f"✅ {doc_id}: {len(pages)} pages processed.")
            else:
                print(f"✅ {doc_id}: Images already exist. Skipping conversion.")
        except Exception as e:
            print(f"❌ Error converting {pdf_path.name}: {e}")

    print("\n--- STEP 2: EXTRACTING TEXT (OCR) ---")
    for doc_folder in OUT_DIR.iterdir():
        if not doc_folder.is_dir(): continue
        
        pages_images = sorted(doc_folder.glob("*.png"))
        print(f"\n📂 Processing Document: {doc_folder.name}")

        for img_path in pages_images:
            json_name = f"{doc_folder.name}_{img_path.stem}_ocr.json"
            
            # باش مايعاودش يدير OCR للملفات لي ديجا خرجات
            if (OCR_OUT_DIR / json_name).exists(): 
                continue

            print(f"  📄 Reading {img_path.name}...")
            img = Image.open(img_path).convert("RGB")
            patches, coords = extract_patches(img)
            
            ocr_texts = ocr_patches(patches)

            ocr_data = {
                "source_pdf": doc_folder.name,
                "page_image": img_path.name,
                "data": []
            }
            
            for i, (bbox, text) in enumerate(zip(coords, ocr_texts)):
                if text and len(text) > 2: 
                    ocr_data["data"].append({
                        "patch_index": i,
                        "bbox": bbox,
                        "text": text
                    })

            with open(OCR_OUT_DIR / json_name, "w", encoding="utf-8") as f:
                json.dump(ocr_data, f, ensure_ascii=False, indent=4)
            print(f"  💾 Saved JSON for {img_path.name}")

    print("\n🎉 All tasks completed successfully!")

process_all_pdfs()

--- Working Directory: c:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject2\chatBotJuridiquesWithRAG- ---

--- STEP 1: CONVERTING 13 PDFs TO IMAGES ---
⏳ Converting AMTJ-parents.pdf...
✅ AMTJ-parents: 2 pages processed.
⏳ Converting cir 01-retraite complémentaire 1.pdf...
✅ cir 01-retraite complémentaire 1: 1 pages processed.
⏳ Converting cir 2-Pèlerinage.pdf...
✅ cir 2-Pèlerinage: 1 pages processed.
⏳ Converting CIR 7-transport aérien.pdf...
✅ CIR 7-transport aérien: 1 pages processed.
⏳ Converting CIR 8-convention transport.pdf...
✅ CIR 8-convention transport: 1 pages processed.
⏳ Converting Cir don femmes enceintes26.pdf...
✅ Cir don femmes enceintes26: 1 pages processed.
⏳ Converting com 06-convention g_Omrane.pdf...
✅ com 06-convention g_Omrane: 1 pages processed.
⏳ Converting com 4 AMC.pdf...
✅ com 4 AMC: 10 pages processed.
⏳ Converting COM 5-séjour linguis.pdf...
✅ COM 5-séjour linguis: 1 pages processed.
⏳ Converting com convention Eqdom.pdf...
✅ com convent

Setup the embedding moudle 

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return a Multilingual embedding model optimized for Arabic.
    """
    model_name = "intfloat/multilingual-e5-base"
    
    model_kwargs = {'device': 'cpu'} 
    encode_kwargs = {'normalize_embeddings': True} 
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embeddings

embedding = download_embeddings()
print("✅ Embeddings model is ready for Arabic!")

C:\Users\vivobook\AppData\Local\Temp\ipykernel_4792\3733159505.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


✅ Embeddings model is ready for Arabic!


In [10]:
vector = embedding.embed_query("الإقتصاد")
print(len(vector))
print(vector)

768
[0.055196527391672134, 0.05592329800128937, 0.013396338559687138, 0.030474532395601273, 0.01198836974799633, -0.05208773910999298, 0.0007374533452093601, -0.040192220360040665, 0.01440285425633192, -0.0037508143577724695, 0.023447250947356224, 0.023675477132201195, 0.15949103236198425, 0.040033288300037384, -0.02711133286356926, -0.04413944110274315, 0.0441996231675148, -0.051685135811567307, 0.055426765233278275, -0.0008227094658650458, 0.054761551320552826, -0.0019403421320021152, 0.042476292699575424, -0.04440610110759735, 0.0009655682952143252, -0.05327524244785309, 0.026688532903790474, 0.007273482158780098, -0.0005164410686120391, 0.03137904033064842, 0.0005725699593313038, -0.04888027161359787, 0.0017705034697428346, 0.0013964641839265823, 0.039450615644454956, 0.07035262137651443, -0.020239785313606262, -0.02500203624367714, 0.07904477417469025, 0.023940294981002808, 0.03044365532696247, 0.023420758545398712, 0.028036922216415405, -0.035178590565919876, 0.028875131160020828

connect to Pinecone DB

In [3]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
print("imports OK")

c:\Users\vivobook\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


In [4]:
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("Key found:", bool(PINECONE_API_KEY))
index_name = "juridique-rag-index"

pc = Pinecone(api_key=PINECONE_API_KEY)
print("Connected!")

Key found: True
Connected!


In [5]:
print(" Loading multilingual-e5-base model...")

embedding = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ Model loaded successfully!")

 Loading multilingual-e5-base model...


C:\Users\vivobook\AppData\Local\Temp\ipykernel_14612\3800351749.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


✅ Model loaded successfully!


In [6]:
if not pc.list_indexes():
    print(f"⏳ Creating a new index called '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=768,  
        metric="cosine"
    )
    print("✅ Index created successfully!")
else:
    existing_indexes = [idx.name for idx in pc.list_indexes()]
    
    if index_name in existing_indexes:
        print(f"✅ Index '{index_name}' already exists. We will use it.")
    else:
        print(f"⏳ Creating a new index called '{index_name}'...")
        pc.create_index(
            name=index_name,
            dimension=768,  
            metric="cosine"
        )
        print("✅ Index created successfully!")

✅ Index 'juridique-rag-index' already exists. We will use it.


In [7]:
current_path = Path.cwd()
BASE_DIR = current_path.parent if current_path.name == "research" else current_path
OCR_DIR = BASE_DIR / "artifacts" / "ocr"

if not OCR_DIR.exists():
    print(f"❌ OCR directory {OCR_DIR} not found. Run process_all_pdfs first.")
    json_files = []
else:
    json_files = list(OCR_DIR.glob("*.json"))

print(f"📂 Found {len(json_files)} JSON files. Reading them...")

documents = []

for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = json.load(f)
        
        for item in content.get('data', []):
            text = item.get('text', '').strip()
            
            if text and len(text) > 2:
                e5_text = f"passage: {text}"
                
                metadata = {
                    "source": content.get('source_pdf', 'Unknown'),
                    "page": content.get('page_image', 'Unknown')
                }
                
                doc = Document(page_content=e5_text, metadata=metadata)
                documents.append(doc)

print(f" Prepared {len(documents)} chunks (paragraphs) ready for Pinecone.")

📂 Found 24 JSON files. Reading them...
 Prepared 329 chunks (paragraphs) ready for Pinecone.


In [ ]:
print(" Uploading vectors to Pinecone... ")
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

print(" DONE! All your legal documents are now vectorized and stored in Pinecone!")

 Uploading vectors to Pinecone... 
 DONE! All your legal documents are now vectorized and stored in Pinecone!


If you want to add  

In [9]:
from langchain_core.documents import Document

dswith = Document(
    page_content="passage: سلام",
    metadata={"source": "me", "page": "none"} 
)

print("⏳ Adding manual document to Pinecone...")
print(docsearch.add_documents(documents=[dswith]))
print("✅ Document added successfully!")

⏳ Adding manual document to Pinecone...
['9b1fc905-9aa2-4465-b5b5-d28a44554368']
✅ Document added successfully!


retriever

In [10]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [11]:
query = "منحة الحج"
retrieved_docs = retriever.get_relevant_documents(query)
retrieved_docs

C:\Users\vivobook\AppData\Local\Temp\ipykernel_14612\3115427282.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(query)


[Document(id='21e82f40-11ae-4f38-9505-653ec94ec60c', metadata={'page': 'page_001.png', 'source': 'cir 2-Pèlerinage'}, page_content='passage: المملحة العربية السعودية و ناريج مغادربها حصريا عبر البوابة الالخترونية للمؤسسة. سادسا : تحدد المنحة المالية للحج برسم السنة المالية 2026 بالنسبة للمنخرطين الذين سيؤدون مناسك الحج بموسم 1447 ه؛ حسب السلم الإداري للمستفيد (الوضعية الحالية بالنسبة للنشيطين أو آخر وضعية إدارية قبل الإحالة على التقاعد'),
 Document(id='a50d59e3-3662-44d6-a4de-8a46f4bf8267', metadata={'page': 'page_001.png', 'source': 'cir 2-Pèlerinage'}, page_content='passage: المملحة العربية السعودية و ناريج مغادربها حصريا عبر البوابة الالخترونية للمؤسسة. سادسا : تحدد المنحة المالية للحج برسم السنة المالية 2026 بالنسبة للمنخرطين الذين سيؤدون مناسك الحج بموسم 1447 ه؛ حسب السلم الإداري للمستفيد (الوضعية الحالية بالنسبة للنشيطين أو آخر وضعية إدارية قبل الإحالة على التقاعد'),
 Document(id='23dab729-d804-4341-a905-b3ce49c2bc10', metadata={'page': 'page_001.png', 'source': 'cir 2-Pèlerin

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("GOOGLE_API_KEY")
print("Key loaded:", key[:10] + "..." if key else "❌ NOT FOUND")

Key loaded: AIzaSyBdPq...


In [15]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
key = os.getenv("GOOGLE_API_KEY")
print("Key:", key[:15] + "..." if key else "❌ NOT FOUND")


Key: AIzaSyBdPqDgWkQ...


augmenting with the LLM

In [16]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

chatModel = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)
system_prompt = (
    "أنت مساعد قانوني خبير (Chatbot Juridique) لتقديم المساعدة والنصائح القانونية. "
    "استخدم فقط المعلومات والنصوص المرفقة في السياق أسفله للإجابة على سؤال المستخدم. "
    "إذا كانت الإجابة غير موجودة في السياق، قل بكل بساطة أنك لا تعرف، إياك أن تخترع أو تؤلف أي معلومات قانونية من عندك. "
    "يجب أن تكون إجابتك واضحة، دقيقة، ومبنية حصرياً على السياق المعطى. "
    "أجب دائماً باللغة العربية.\n\n"
    "السياق:\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "query: منحة الحج "})

print("الجواب:")
print(response["answer"])

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2 seconds as it raised NotFound: 404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods..


ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 API key expired. Please renew the API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key expired. Please renew the API key."
]